# exp097_modelpkg_tiny_gate_on_exp073 inference

exp073 inference prediction に selected model-package tiny gate を適用し、guard 通過時だけ `submission.csv` を書く。

## Contents

1. Setup and configuration
2. Guarded tiny gate inference
3. Submission contract
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, load_config
from modelpkg_tiny_gate_on_exp073 import run_inference_from_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print('Experiment:', EXPERIMENT_NAME)
print('Route:', config['experiment']['route'])
print('Sample submission:', paths.sample_submission_path)
print('Submission output:', paths.submission_path)
print('Artifacts:', paths.artifacts_dir)
print('Selected variant:', config['inference']['selected_variant'])
print('Generate exp073 base on current test:', config['inference'].get('generate_exp073_base_on_current_test'))
print('Disable modelpkg on row mismatch:', config['inference'].get('disable_modelpkg_on_row_mismatch'))
print('Guard:', config['audit']['selection_guard'])

## 2. Guarded tiny gate inference

This cell runs the same grid audit as train and writes the selected candidate only when all configured guards pass.

In [ ]:
summary = run_inference_from_config(
    config,
    sample_path=paths.sample_submission_path,
    output_dir=paths.artifacts_dir,
    submission_path=paths.submission_path,
)
print('Status:', summary['status'])
print('Selected variant:', summary.get('selected_variant'))
print('Selected passes all guards:', summary.get('selected_passes_all_guards'))
print(json.dumps(summary.get('selected_summary', {}), indent=2))
if summary.get('reason'):
    print('Reason:', summary['reason'])

## 3. Submission contract


In [ ]:
valid_statuses = {'inference_submission_written', 'modelpkg_disabled_base_submission_written'}
if summary['status'] not in valid_statuses:
    raise RuntimeError(f"No submission was written because status={summary['status']}")

sample = pd.read_csv(paths.sample_submission_path)
submission = pd.read_csv(paths.submission_path)
if list(submission.columns) != ['id', 'tvt']:
    raise RuntimeError(f"Unexpected submission columns: {list(submission.columns)}")
if len(submission) != len(sample):
    raise RuntimeError(f"Submission row mismatch: {len(submission)} vs {len(sample)}")
if not submission['id'].astype(str).equals(sample['id'].astype(str)):
    raise RuntimeError('Submission ids do not match sample_submission order')
if submission['tvt'].isna().any():
    raise RuntimeError('Submission contains missing tvt')
display(submission.head())
display(submission['tvt'].describe().to_frame('tvt'))

## 4. Metrics and artifacts


In [ ]:
metrics_path = summary.get('artifacts', {}).get('variant_metrics')
if metrics_path:
    metrics = pd.read_csv(metrics_path)
    display(metrics)
else:
    metrics = None
    print('Variant metrics unavailable:', summary.get('reason'))

run_metrics = {
    'experiment': EXPERIMENT_NAME,
    'status': summary['status'],
    'route': config['experiment']['route'],
    'cv': None,
    'public_lb': None,
    'private_lb': None,
    'selected_variant': summary.get('selected_variant'),
    'selected_passes_all_guards': summary.get('selected_passes_all_guards'),
    'selected_summary': summary.get('selected_summary'),
    'reason': summary.get('reason'),
    'submission': summary.get('submission'),
    'artifacts': summary['artifacts'],
}
paths.metrics_path.write_text(json.dumps(run_metrics, indent=2, sort_keys=True) + '\n')
print('Metrics written:', paths.metrics_path)
print('Submission:', paths.submission_path)